In [1]:
from pathlib import Path
import duckdb
import os

PROJECT_ROOT = Path(r"C:\Users\USUARIO WINDOWS\CODIGOS\PAZ CD").resolve()
os.chdir(PROJECT_ROOT)

DB_PATH  = PROJECT_ROOT / "data" / "exoplanets.duckdb"
RAW_CSV  = PROJECT_ROOT / "data" / "raw" / "pscomppars.csv"
ART_DIR  = PROJECT_ROOT / "artifacts"
DATA_DIR = PROJECT_ROOT / "data"

ART_DIR.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(DB_PATH))

def sql_path(p: Path) -> str:
    return "'" + p.resolve().as_posix().replace("'", "''") + "'"

con.execute("DROP VIEW IF EXISTS raw_ps")
con.execute(f"CREATE VIEW raw_ps AS SELECT * FROM read_csv_auto({sql_path(RAW_CSV)})")

tables = [r[0] for r in con.sql("SHOW TABLES").fetchall()]
print("Tablas en DB:", tables)

assert "silver_planet_v3" in tables or "silver_planet" in tables, "Ejecuta W09 primero para tener silver_planet_v3"

print("Setup OK ✅")

Tablas en DB: ['demo_numbers', 'dim_discovery', 'dim_host_bad', 'dim_host_fixed', 'dim_host_full', 'dim_host_ra', 'dim_host_sk', 'fact_planet', 'fact_planet_raw', 'fact_planet_sk', 'quality_w03a', 'raw_ps', 'silver_planet', 'students', 'submissions']
Setup OK ✅


## particionar por `disc_era`

In [2]:
tables = [r[0] for r in con.sql("SHOW TABLES").fetchall()]
src = "silver_planet_v3" if "silver_planet_v3" in tables else "silver_planet"

PARQUET_DIR = DATA_DIR / "partitioned"
PARQUET_DIR.mkdir(parents=True, exist_ok=True)

# En esta versión calculamos disc_era directamente para evitar depender de si ya existe
if src == "silver_planet_v3":
    query = f"""
    SELECT
      *,
      CASE
        WHEN disc_year_int IS NULL THEN 'unknown'
        WHEN disc_year_int < 2000 THEN 'pre_2000'
        WHEN disc_year_int BETWEEN 2000 AND 2009 THEN '2000s'
        WHEN disc_year_int BETWEEN 2010 AND 2019 THEN '2010s'
        WHEN disc_year_int >= 2020 THEN '2020s'
        ELSE 'unknown'
      END AS disc_era
    FROM {src}
    """
else:
    query = f"""
    SELECT
      *,
      CASE
        WHEN disc_year IS NULL THEN 'unknown'
        WHEN disc_year < 2000 THEN 'pre_2000'
        WHEN disc_year BETWEEN 2000 AND 2009 THEN '2000s'
        WHEN disc_year BETWEEN 2010 AND 2019 THEN '2010s'
        WHEN disc_year >= 2020 THEN '2020s'
        ELSE 'unknown'
      END AS disc_era
    FROM {src}
    """

parquet_path = str((PARQUET_DIR / "silver_v3_partitioned").resolve()).replace("\\", "/")

con.execute(f"""
COPY (
  {query}
)
TO '{parquet_path}'
(FORMAT PARQUET, PARTITION_BY (disc_era), OVERWRITE_OR_IGNORE TRUE)
""")

print("Particionamiento completado ✅")
print("Ruta:", parquet_path)

Particionamiento completado ✅
Ruta: C:/Users/USUARIO WINDOWS/CODIGOS/PAZ CD/data/partitioned/silver_v3_partitioned


## Número de archivos por partición

In [3]:
parquet_base = PARQUET_DIR / "silver_v3_partitioned"

partition_dirs = sorted([d for d in parquet_base.iterdir() if d.is_dir()])

print(f"Particiones creadas: {len(partition_dirs)}")
print("-" * 50)

for pdir in partition_dirs:
    files = list(pdir.glob("*.parquet"))
    total_bytes = sum(f.stat().st_size for f in files)
    print(f"{pdir.name:30s} {len(files)} archivo(s) {total_bytes/1024:.1f} KB")

Particiones creadas: 5
--------------------------------------------------
disc_era=2000s                 1 archivo(s) 27.8 KB
disc_era=2010s                 1 archivo(s) 215.5 KB
disc_era=2020s                 1 archivo(s) 142.8 KB
disc_era=pre_2000              1 archivo(s) 4.8 KB
disc_era=unknown               1 archivo(s) 2.0 KB


## resumen por particion

In [7]:
glob_pattern = str(parquet_base.resolve()).replace("\\", "/") + "/**/*.parquet"

con.sql(f"""
SELECT
  disc_era,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 2) AS avg_radius,
  ROUND(AVG(pl_bmasse), 2) AS avg_mass
FROM read_parquet('{glob_pattern}', hive_partitioning=true)
GROUP BY disc_era
ORDER BY disc_era
""").show()

┌──────────┬───────────┬────────────┬──────────┐
│ disc_era │ n_planets │ avg_radius │ avg_mass │
│ varchar  │   int64   │   double   │  double  │
├──────────┼───────────┼────────────┼──────────┤
│ 2000s    │       378 │      12.33 │  1145.78 │
│ 2010s    │      3681 │       4.89 │   268.55 │
│ 2020s    │      2196 │       6.06 │   476.05 │
│ pre_2000 │        30 │      12.18 │  1075.68 │
│ unknown  │         1 │       2.86 │      5.0 │
└──────────┴───────────┴────────────┴──────────┘



## Evidencia de pruning con `EXPLAIN ANALYZE`

In [8]:
explain_q = f"""
EXPLAIN ANALYZE
SELECT
  disc_era,
  COUNT(*) AS n_planets,
  ROUND(AVG(pl_rade), 2) AS avg_radius
FROM read_parquet('{glob_pattern}', hive_partitioning=true)
WHERE disc_era = '2020s'
GROUP BY disc_era
"""

result = con.sql(explain_q).fetchall()
explain_text = "\n".join(r[1] for r in result)

print(explain_text)

out_explain = ART_DIR / "w10b_explain_analyze_pruning.txt"
out_explain.write_text(explain_text, encoding="utf-8")

print("Guardado en:", out_explain)

┌─────────────────────────────────────┐
│┌───────────────────────────────────┐│
││    Query Profiling Information    ││
│└───────────────────────────────────┘│
└─────────────────────────────────────┘
 EXPLAIN ANALYZE SELECT   disc_era,   COUNT(*) AS n_planets,   ROUND(AVG(pl_rade), 2) AS avg_radius FROM read_parquet('C:/Users/USUARIO WINDOWS/CODIGOS/PAZ CD/data/partitioned/silver_v3_partitioned/**/*.parquet', hive_partitioning=true) WHERE disc_era = '2020s' GROUP BY disc_era 
┌────────────────────────────────────────────────┐
│┌──────────────────────────────────────────────┐│
││              Total Time: 0.0164s             ││
│└──────────────────────────────────────────────┘│
└────────────────────────────────────────────────┘
┌───────────────────────────┐
│           QUERY           │
└─────────────┬─────────────┘
┌─────────────┴─────────────┐
│      EXPLAIN_ANALYZE      │
│    ────────────────────   │
│           0 rows          │
│          (0.00s)          │
└─────────────┬─────────

In [9]:
total_all = con.sql(f"""
SELECT COUNT(*) AS n
FROM read_parquet('{glob_pattern}', hive_partitioning=true)
""").fetchone()[0]

total_2020s = con.sql(f"""
SELECT COUNT(*) AS n
FROM read_parquet('{glob_pattern}', hive_partitioning=true)
WHERE disc_era = '2020s'
""").fetchone()[0]

print("Total filas:", total_all)
print("Filas en 2020s:", total_2020s)
print(f"Reducción aproximada: {(1 - total_2020s / total_all) * 100:.1f}%")

Total filas: 6286
Filas en 2020s: 2196
Reducción aproximada: 65.1%
